# Inspect Political-Corruption Topic Results

This notebook reads the final topic-model outputs and GPT topic labels, then produces human-readable tables and interactive country/time visualizations.

Expected inputs from the topic workflow:

- `topic_info.csv`
- `document_topics.csv.gz`
- `topic_labels_llm.csv`

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [NOTEBOOK_PATH, *NOTEBOOK_PATH.parents] if (path / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

DEFAULT_TOPIC_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "topic_classification/bertopic_political_corruption_granular"
)

TOPIC_DIR = DEFAULT_TOPIC_DIR
TOP_N_TOPICS = 12
TIME_UNIT = "year"  # "year" or "month"
ANALYSIS_LABEL_COLUMN = "topic_group_short_label"  # Use "topic_short_label" for fine-grained topics.
RAW_TOPIC_ID_COLUMN = "topic"
RAW_TOPIC_LABEL_COLUMN = "topic_short_label"
LIFT_FLOOR = 0.005  # Hide labels with very small baseline shares in lift plots.

TOPIC_INFO_PATH = TOPIC_DIR / "topic_info.csv"
DOCUMENT_TOPICS_PATH = TOPIC_DIR / "document_topics.csv.gz"
TOPIC_LABELS_PATH = TOPIC_DIR / "topic_labels_llm.csv"
TOPIC_GROUPS_PATH = TOPIC_DIR / "topic_groups_llm.csv"

TOPIC_DIR

## Load Results

In [ ]:
missing = [path for path in [TOPIC_INFO_PATH, DOCUMENT_TOPICS_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError("Missing topic output(s): " + ", ".join(str(path) for path in missing))

topic_info = pd.read_csv(TOPIC_INFO_PATH)
document_topics = pd.read_csv(DOCUMENT_TOPICS_PATH)

if TOPIC_LABELS_PATH.exists():
    topic_labels = pd.read_csv(TOPIC_LABELS_PATH)
else:
    topic_labels = pd.DataFrame()
    print(f"No GPT topic labels found yet: {TOPIC_LABELS_PATH}")

if TOPIC_GROUPS_PATH.exists():
    topic_groups = pd.read_csv(TOPIC_GROUPS_PATH)
else:
    topic_groups = pd.DataFrame()
    print(f"No GPT topic groups found yet: {TOPIC_GROUPS_PATH}")

print(f"Topic directory: {TOPIC_DIR}")
print(f"Topics: {len(topic_info):,}")
print(f"Document-topic rows: {len(document_topics):,}")
print(f"GPT-labelled topics: {len(topic_labels):,}")
print(f"GPT-grouped topics: {len(topic_groups):,}")

In [ ]:
def build_topic_lookup(topic_info, topic_labels, topic_groups):
    labels = topic_info[["Topic", "Name", "Count"]].copy()
    if not topic_labels.empty:
        keep_cols = [
            col for col in [
                "Topic",
                "llm_topic_label",
                "llm_topic_short_label",
                "llm_primary_domain",
                "llm_secondary_domain",
                "llm_generic_domain_label",
                "llm_generic_domain_short_label",
                "llm_corruption_type",
                "llm_country_event_specific",
                "llm_domain_evidence",
                "llm_cross_country_comparability",
                "llm_label_rationale",
                "llm_topic_summary",
                "llm_inclusion_rule",
                "llm_exclusion_rule",
                "llm_confidence",
            ]
            if col in topic_labels.columns
        ]
        labels = labels.merge(topic_labels[keep_cols], on="Topic", how="left")

    labels["topic_label"] = labels.get("llm_topic_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_label"] = labels["topic_label"].fillna("").astype(str)
    labels.loc[labels["topic_label"].str.strip().eq(""), "topic_label"] = labels["Name"]

    labels["topic_short_label"] = labels.get("llm_topic_short_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_short_label"] = labels["topic_short_label"].fillna("").astype(str)
    labels.loc[labels["topic_short_label"].str.strip().eq(""), "topic_short_label"] = labels["topic_label"]

    labels["primary_domain"] = labels.get("llm_primary_domain", pd.Series(index=labels.index, dtype=object))
    labels["primary_domain"] = labels["primary_domain"].fillna("").astype(str)
    labels.loc[labels["primary_domain"].str.strip().eq(""), "primary_domain"] = labels.get("llm_corruption_type", labels["topic_label"])
    labels["primary_domain"] = labels["primary_domain"].fillna(labels["topic_label"])

    labels["secondary_domain"] = labels.get("llm_secondary_domain", pd.Series(index=labels.index, dtype=object))
    labels["secondary_domain"] = labels["secondary_domain"].fillna("none").astype(str)

    labels["generic_domain_label"] = labels.get("llm_generic_domain_label", pd.Series(index=labels.index, dtype=object))
    labels["generic_domain_label"] = labels["generic_domain_label"].fillna("").astype(str)
    labels.loc[labels["generic_domain_label"].str.strip().eq(""), "generic_domain_label"] = labels["primary_domain"]
    labels["generic_domain_label"] = labels["generic_domain_label"].fillna(labels["topic_label"])

    labels["generic_domain_short_label"] = labels.get("llm_generic_domain_short_label", pd.Series(index=labels.index, dtype=object))
    labels["generic_domain_short_label"] = labels["generic_domain_short_label"].fillna("").astype(str)
    labels.loc[labels["generic_domain_short_label"].str.strip().eq(""), "generic_domain_short_label"] = labels["generic_domain_label"]

    if not topic_groups.empty:
        group_cols = [
            col for col in [
                "Topic",
                "topic_group_id",
                "topic_group_label",
                "topic_group_short_label",
                "topic_group_summary",
                "topic_group_cross_country_comparability",
                "topic_grouping_principle",
                "topic_group_assignment_rationale",
            ]
            if col in topic_groups.columns
        ]
        labels = labels.merge(topic_groups[group_cols], on="Topic", how="left")

    labels["topic_group_label"] = labels.get("topic_group_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_group_label"] = labels["topic_group_label"].fillna(labels["topic_label"])
    labels["topic_group_short_label"] = labels.get("topic_group_short_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_group_short_label"] = labels["topic_group_short_label"].fillna(labels["topic_short_label"])
    return labels

topic_lookup = build_topic_lookup(topic_info, topic_labels, topic_groups)
docs = document_topics.merge(topic_lookup, left_on="topic", right_on="Topic", how="left")
docs["topic_label"] = docs["topic_label"].fillna(docs["topic"].astype(str))
docs["topic_short_label"] = docs["topic_short_label"].fillna(docs["topic_label"])
for optional_col, fallback_col in [
    ("primary_domain", "topic_label"),
    ("secondary_domain", None),
    ("generic_domain_label", "topic_label"),
    ("generic_domain_short_label", "topic_short_label"),
    ("topic_group_label", "topic_label"),
    ("topic_group_short_label", "topic_short_label"),
]:
    if optional_col not in docs.columns:
        docs[optional_col] = "none" if fallback_col is None else docs[fallback_col]
    elif fallback_col is None:
        docs[optional_col] = docs[optional_col].fillna("none")
    else:
        docs[optional_col] = docs[optional_col].fillna(docs[fallback_col])
if ANALYSIS_LABEL_COLUMN not in docs.columns:
    raise ValueError(f"ANALYSIS_LABEL_COLUMN not found: {ANALYSIS_LABEL_COLUMN}")
if "analysis_weight" not in docs.columns:
    docs["analysis_weight"] = 1.0

non_outlier_docs = docs[docs["topic"].ne(-1)].copy()
topic_lookup.head()

## Substantive Summary

This section gives a readable substantive overview of the topic solution currently selected by `ANALYSIS_LABEL_COLUMN`. It is meant as the first interpretation layer before looking at detailed lift plots and example articles.


In [ ]:
def top_values(series, n=3):
    values = series.dropna().astype(str)
    values = values[values.str.strip().ne("")]
    return "; ".join(values.value_counts().head(n).index.tolist())

summary_base = (
    non_outlier_docs.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)
    .agg(
        weighted_articles=("analysis_weight", "sum"),
        raw_topics=("topic", "nunique"),
        top_fine_topics=("topic_short_label", lambda s: top_values(s, 5)),
    )
    .reset_index()
)
summary_base["weighted_share"] = summary_base["weighted_articles"] / summary_base["weighted_articles"].sum()

summary_country = (
    non_outlier_docs.groupby([ANALYSIS_LABEL_COLUMN, "country"], dropna=False)["analysis_weight"]
    .sum()
    .rename("country_weighted_articles")
    .reset_index()
)
summary_country["topic_total"] = summary_country.groupby(ANALYSIS_LABEL_COLUMN)["country_weighted_articles"].transform("sum")
summary_country["country_share_within_topic"] = summary_country["country_weighted_articles"] / summary_country["topic_total"]

country_diagnostics = (
    summary_country.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)
    .agg(
        max_country_share=("country_share_within_topic", "max"),
        countries_ge_5pct=("country_share_within_topic", lambda s: int((s >= 0.05).sum())),
    )
    .reset_index()
)

top_country_text = (
    summary_country.sort_values([ANALYSIS_LABEL_COLUMN, "country_weighted_articles"], ascending=[True, False])
    .groupby(ANALYSIS_LABEL_COLUMN)
    .head(3)
    .assign(country_piece=lambda d: d["country"] + " (" + (100 * d["country_share_within_topic"]).round(1).astype(str) + "%)")
    .groupby(ANALYSIS_LABEL_COLUMN)["country_piece"]
    .apply(lambda s: "; ".join(s))
    .rename("top_countries")
    .reset_index()
)

substantive_summary = (
    summary_base.merge(country_diagnostics, on=ANALYSIS_LABEL_COLUMN, how="left")
    .merge(top_country_text, on=ANALYSIS_LABEL_COLUMN, how="left")
)

if "topic_group_summary" in non_outlier_docs.columns:
    group_descriptions = (
        non_outlier_docs.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)
        .agg(
            substantive_description=("topic_group_summary", lambda s: top_values(s, 1)),
            group_comparability=("topic_group_cross_country_comparability", lambda s: top_values(s, 1))
        )
        .reset_index()
    )
    substantive_summary = substantive_summary.merge(group_descriptions, on=ANALYSIS_LABEL_COLUMN, how="left")
elif "llm_topic_summary" in non_outlier_docs.columns:
    topic_descriptions = (
        non_outlier_docs.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)
        .agg(
            substantive_description=("llm_topic_summary", lambda s: top_values(s, 1)),
            group_comparability=("llm_cross_country_comparability", lambda s: top_values(s, 1))
        )
        .reset_index()
    )
    substantive_summary = substantive_summary.merge(topic_descriptions, on=ANALYSIS_LABEL_COLUMN, how="left")

substantive_summary = substantive_summary.sort_values("weighted_articles", ascending=False)

summary_display_cols = [
    ANALYSIS_LABEL_COLUMN,
    "weighted_articles",
    "weighted_share",
    "raw_topics",
    "countries_ge_5pct",
    "max_country_share",
    "top_countries",
    "top_fine_topics",
    "substantive_description",
    "group_comparability",
]
summary_display_cols = [col for col in summary_display_cols if col in substantive_summary.columns]

display(substantive_summary[summary_display_cols].head(30))

n_units = len(substantive_summary)
concentrated = int(substantive_summary["max_country_share"].ge(0.70).sum())
cross_country = int(substantive_summary["countries_ge_5pct"].ge(3).sum())
top_unit = substantive_summary.iloc[0]

summary_text = f"""
### Quick Read

- The current analysis level is `{ANALYSIS_LABEL_COLUMN}` and contains **{n_units}** plotted topics/groups.
- The largest topic/group is **{top_unit[ANALYSIS_LABEL_COLUMN]}**, accounting for **{top_unit['weighted_share']:.1%}** of weighted articles.
- **{concentrated}** topics/groups are strongly country-concentrated (`max_country_share >= 70%`).
- **{cross_country}** topics/groups have meaningful presence in at least three countries (`countries_ge_5pct >= 3`).

Use this as a substantive map, then use the lift sections below to see where each topic/group is unusually prominent by country or over time.
"""
display(Markdown(summary_text))


## Grouping Explanation

When `topic_groups_llm.csv` is available, this section explains how fine-grained BERTopic topics were grouped into higher-order inductive groups. Use it to audit whether the grouping is substantively intuitive.


In [ ]:
if "topic_group_short_label" not in topic_lookup.columns or topic_groups.empty:
    print("No topic-group file loaded. Run group_topics_with_llm.py, or inspect fine-grained topics with ANALYSIS_LABEL_COLUMN = 'topic_short_label'.")
else:
    group_overview = (
        topic_lookup[topic_lookup["Topic"].ne(-1)]
        .groupby(["topic_group_id", "topic_group_short_label", "topic_group_label"], dropna=False)
        .agg(
            group_weighted_topics=("Count", "sum"),
            fine_topics=("Topic", "nunique"),
            grouping_principle=("topic_grouping_principle", lambda s: top_values(s, 1)),
            group_summary=("topic_group_summary", lambda s: top_values(s, 1)),
            comparability=("topic_group_cross_country_comparability", lambda s: top_values(s, 1)),
            component_topics=("topic_short_label", lambda s: top_values(s, 8)),
        )
        .reset_index()
        .sort_values("group_weighted_topics", ascending=False)
    )
    display(group_overview)

    assignment_cols = [
        "Topic",
        "topic_short_label",
        "topic_label",
        "topic_group_short_label",
        "topic_group_label",
        "topic_group_assignment_rationale",
        "llm_topic_summary",
        "Count",
    ]
    assignment_cols = [col for col in assignment_cols if col in topic_lookup.columns]
    topic_assignment_explanation = (
        topic_lookup[topic_lookup["Topic"].ne(-1)]
        [assignment_cols]
        .sort_values(["topic_group_short_label", "Count"], ascending=[True, False])
    )
    display(topic_assignment_explanation.head(100))


## Topic Or Group Overview

In [ ]:
topic_totals = (
    non_outlier_docs.groupby([ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
    .sort_values("weighted_articles", ascending=False)
)
topic_totals["weighted_share"] = topic_totals["weighted_articles"] / topic_totals["weighted_articles"].sum()
top_labels = topic_totals.head(TOP_N_TOPICS)[ANALYSIS_LABEL_COLUMN].tolist()

display(topic_totals.head(30))

fig = px.bar(
    topic_totals.head(30).sort_values("weighted_articles"),
    x="weighted_articles",
    y=ANALYSIS_LABEL_COLUMN,
    orientation="h",
    title="Largest Political-Corruption Topics",
    labels={"weighted_articles": "Weighted articles", ANALYSIS_LABEL_COLUMN: "Topic"},
)
fig.update_layout(height=800)
fig.show()

In [ ]:
summary_cols = [
    col for col in [
        "Topic",
        "Count",
        "topic_label",
        "primary_domain",
        "secondary_domain",
        "topic_short_label",
        "topic_group_label",
        "topic_group_short_label",
        "topic_group_summary",
        "topic_grouping_principle",
        "topic_group_cross_country_comparability",
        "topic_group_assignment_rationale",
        "llm_cross_country_comparability",
        "llm_country_event_specific",
        "llm_label_rationale",
        "llm_topic_summary",
        "generic_domain_label",
        "generic_domain_short_label",
        "llm_corruption_type",
        "llm_domain_evidence",
        "llm_inclusion_rule",
        "llm_exclusion_rule",
        "llm_confidence",
        "Name",
    ]
    if col in topic_lookup.columns
]

display(
    topic_lookup[topic_lookup["Topic"].ne(-1)]
    .sort_values("Count", ascending=False)
    [summary_cols]
    .head(30)
)

## Cross-Country Topic Composition

In [ ]:
docs_top = non_outlier_docs[non_outlier_docs[ANALYSIS_LABEL_COLUMN].isin(top_labels)].copy()

country_topic = (
    docs_top.groupby(["country", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
country_topic["share"] = country_topic["weighted_articles"] / country_topic.groupby("country")["weighted_articles"].transform("sum")

heatmap_data = country_topic.pivot(index="country", columns=ANALYSIS_LABEL_COLUMN, values="share").fillna(0)
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"color": "Within-country share"},
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topics By Country",
)
fig.update_layout(height=650)
fig.show()

## Country Topic Lift Diagnostics

Shares can look flat when topic prevalence is similar. Lift shows where a country over- or under-represents a topic compared with the overall sample baseline. Values are log2 lift: `1` means twice the baseline share, `-1` means half the baseline share.

In [ ]:
overall_label_share = (
    docs_top.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)["analysis_weight"]
    .sum()
    .div(docs_top["analysis_weight"].sum())
    .rename("overall_share")
    .reset_index()
)

country_topic_lift = country_topic.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
country_topic_lift = country_topic_lift[country_topic_lift["overall_share"].ge(LIFT_FLOOR)].copy()
country_topic_lift["lift"] = country_topic_lift["share"] / country_topic_lift["overall_share"]
country_topic_lift["log2_lift"] = np.log2(country_topic_lift["lift"].replace(0, np.nan))

lift_heatmap = country_topic_lift.pivot(index="country", columns=ANALYSIS_LABEL_COLUMN, values="log2_lift").fillna(0)
fig = px.imshow(
    lift_heatmap,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    labels={"color": "log2 lift"},
    title=f"Country Over/Under-Representation Of Top {TOP_N_TOPICS} Topics",
)
fig.update_layout(height=650)
fig.show()

display(
    country_topic_lift.sort_values("log2_lift", ascending=False)
    [["country", ANALYSIS_LABEL_COLUMN, "share", "overall_share", "lift", "log2_lift", "weighted_articles"]]
    .head(30)
)


## Topic Shares Over Time

In [ ]:
trend_docs = docs_top.copy()

if TIME_UNIT == "month":
    if "date_parsed" not in trend_docs.columns:
        raise ValueError("Monthly trends require date_parsed in document_topics.csv.gz.")
    trend_docs["date_parsed"] = pd.to_datetime(trend_docs["date_parsed"], errors="coerce", utc=True)
    trend_docs["period"] = trend_docs["date_parsed"].dt.tz_convert(None).dt.to_period("M").astype(str)
else:
    trend_docs["period"] = trend_docs["year"].astype("Int64").astype(str)

time_topic = (
    trend_docs.groupby(["period", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
time_topic["share"] = time_topic["weighted_articles"] / time_topic.groupby("period")["weighted_articles"].transform("sum")

fig = px.area(
    time_topic.sort_values("period"),
    x="period",
    y="share",
    color=ANALYSIS_LABEL_COLUMN,
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topic Shares Over Time",
    labels={"period": TIME_UNIT.title(), "share": "Topic share", ANALYSIS_LABEL_COLUMN: "Topic"},
)
fig.update_layout(height=650, hovermode="x unified")
fig.show()

## Time Topic Lift Diagnostics

This shows which topics become unusually prominent in particular years or months, relative to their overall baseline.

In [ ]:
time_topic_lift = time_topic.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
time_topic_lift = time_topic_lift[time_topic_lift["overall_share"].ge(LIFT_FLOOR)].copy()
time_topic_lift["lift"] = time_topic_lift["share"] / time_topic_lift["overall_share"]
time_topic_lift["log2_lift"] = np.log2(time_topic_lift["lift"].replace(0, np.nan))

time_lift_heatmap = time_topic_lift.pivot(index="period", columns=ANALYSIS_LABEL_COLUMN, values="log2_lift").fillna(0)
fig = px.imshow(
    time_lift_heatmap,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    labels={"color": "log2 lift"},
    title=f"Over/Under-Representation Of Top {TOP_N_TOPICS} Topics Over Time",
)
fig.update_layout(height=650)
fig.show()

display(
    time_topic_lift.sort_values("log2_lift", ascending=False)
    [["period", ANALYSIS_LABEL_COLUMN, "share", "overall_share", "lift", "log2_lift", "weighted_articles"]]
    .head(30)
)


## Country Trends Over Time

In [ ]:
country_time = (
    trend_docs.groupby(["country", "period", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)

fig = px.line(
    country_time.sort_values("period"),
    x="period",
    y="weighted_articles",
    color=ANALYSIS_LABEL_COLUMN,
    facet_row="country",
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topic Volume By Country Over Time",
    labels={"period": TIME_UNIT.title(), "weighted_articles": "Weighted articles", ANALYSIS_LABEL_COLUMN: "Topic"},
    height=1400,
)
fig.update_yaxes(matches=None)
fig.show()

## Raw Topic Specificity Diagnostics

These tables flag clusters dominated by one country and show how raw BERTopic clusters map onto the plotted inductive topic labels.

In [ ]:
raw_group_cols = [RAW_TOPIC_ID_COLUMN, RAW_TOPIC_LABEL_COLUMN]
raw_country = (
    non_outlier_docs.groupby(raw_group_cols + ["country"], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
raw_country["topic_total"] = raw_country.groupby(raw_group_cols, dropna=False)["weighted_articles"].transform("sum")
raw_country["country_share_within_raw_topic"] = raw_country["weighted_articles"] / raw_country["topic_total"]
raw_specificity = (
    raw_country.groupby(raw_group_cols, dropna=False)
    .agg(
        weighted_articles=("topic_total", "first"),
        max_country_share=("country_share_within_raw_topic", "max"),
        countries_ge_5pct=("country_share_within_raw_topic", lambda s: int((s >= 0.05).sum())),
    )
    .reset_index()
    .sort_values("max_country_share", ascending=False)
)

display(raw_specificity.head(30))

raw_to_label_group_cols = list(raw_group_cols)
if ANALYSIS_LABEL_COLUMN not in raw_to_label_group_cols:
    raw_to_label_group_cols.append(ANALYSIS_LABEL_COLUMN)

raw_to_label = (
    non_outlier_docs.groupby(raw_to_label_group_cols, dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
    .sort_values("weighted_articles", ascending=False)
)
display(raw_to_label.head(50))


## Country-Time Lift

This is the strongest variance check: within each country-year, which topics are unusually prominent relative to their overall baseline?

In [ ]:
country_time_share = country_time.copy()
country_time_share["share"] = country_time_share["weighted_articles"] / country_time_share.groupby(["country", "period"])["weighted_articles"].transform("sum")
country_time_lift = country_time_share.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
country_time_lift = country_time_lift[country_time_lift["overall_share"].ge(LIFT_FLOOR)].copy()
country_time_lift["lift"] = country_time_lift["share"] / country_time_lift["overall_share"]
country_time_lift["log2_lift"] = np.log2(country_time_lift["lift"].replace(0, np.nan))

fig = px.line(
    country_time_lift.sort_values("period"),
    x="period",
    y="log2_lift",
    color=ANALYSIS_LABEL_COLUMN,
    facet_row="country",
    title=f"Country-Time Topic Lift For Top {TOP_N_TOPICS} Topics",
    labels={"period": TIME_UNIT.title(), "log2_lift": "log2 lift", ANALYSIS_LABEL_COLUMN: "Topic"},
    height=1400,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_yaxes(matches=None)
fig.show()


## Inspect Example Articles By Topic

In [ ]:
SELECT_LABEL = top_labels[0] if top_labels else None
N_EXAMPLES = 10

example_cols = [col for col in ["country", "year", "date_parsed", "prob_political_corruption", "topic_label", "article_text"] if col in docs.columns]

if SELECT_LABEL is not None:
    display(
        docs[docs[ANALYSIS_LABEL_COLUMN].eq(SELECT_LABEL)]
        .sort_values("analysis_weight", ascending=False)
        [example_cols]
        .head(N_EXAMPLES)
    )
else:
    print("No non-outlier topics available.")

## Export Summary Tables

In [ ]:
EXPORT_DIR = TOPIC_DIR / "inspection_tables"
LATEX_DIR = EXPORT_DIR / "latex"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LATEX_DIR.mkdir(parents=True, exist_ok=True)

substantive_summary.to_csv(EXPORT_DIR / "substantive_topic_summary.csv", index=False)
if "group_overview" in globals():
    group_overview.to_csv(EXPORT_DIR / "topic_group_overview.csv", index=False)
if "topic_assignment_explanation" in globals():
    topic_assignment_explanation.to_csv(EXPORT_DIR / "topic_group_assignment_explanation.csv", index=False)
topic_totals.to_csv(EXPORT_DIR / "topic_weighted_totals.csv", index=False)
country_topic.to_csv(EXPORT_DIR / "country_topic_shares_top_topics.csv", index=False)
time_topic.to_csv(EXPORT_DIR / "topic_shares_over_time_top_topics.csv", index=False)
country_time.to_csv(EXPORT_DIR / "country_topic_trends_top_topics.csv", index=False)
country_topic_lift.to_csv(EXPORT_DIR / "country_topic_lift_top_topics.csv", index=False)
time_topic_lift.to_csv(EXPORT_DIR / "topic_lift_over_time_top_topics.csv", index=False)
country_time_lift.to_csv(EXPORT_DIR / "country_time_topic_lift_top_topics.csv", index=False)
raw_specificity.to_csv(EXPORT_DIR / "raw_topic_country_specificity.csv", index=False)
raw_to_label.to_csv(EXPORT_DIR / "raw_topic_to_label_mapping.csv", index=False)


def percent_string(series, digits=1):
    return (100 * series.astype(float)).round(digits).astype(str) + "%"


def weighted_count_string(series, digits=0):
    return series.astype(float).round(digits).map(lambda value: f"{value:,.0f}")


def weighted_topic_table():
    weighted = (
        non_outlier_docs.groupby("topic", dropna=False)["analysis_weight"]
        .sum()
        .rename("weighted_articles")
        .reset_index()
    )
    weighted["weighted_share"] = weighted["weighted_articles"] / weighted["weighted_articles"].sum()

    country_by_topic = (
        non_outlier_docs.groupby(["topic", "country"], dropna=False)["analysis_weight"]
        .sum()
        .rename("country_weighted_articles")
        .reset_index()
    )
    country_by_topic["topic_total"] = country_by_topic.groupby("topic")["country_weighted_articles"].transform("sum")
    country_by_topic["country_share_within_topic"] = country_by_topic["country_weighted_articles"] / country_by_topic["topic_total"]
    diagnostics = (
        country_by_topic.groupby("topic", dropna=False)
        .agg(
            max_country_share=("country_share_within_topic", "max"),
            countries_ge_5pct=("country_share_within_topic", lambda s: int((s >= 0.05).sum())),
        )
        .reset_index()
    )
    top_countries = (
        country_by_topic.sort_values(["topic", "country_weighted_articles"], ascending=[True, False])
        .groupby("topic")
        .head(3)
        .assign(country_piece=lambda d: d["country"] + " (" + percent_string(d["country_share_within_topic"], 1) + ")")
        .groupby("topic")["country_piece"]
        .apply(lambda s: "; ".join(s))
        .rename("top_countries")
        .reset_index()
    )
    return weighted.merge(diagnostics, on="topic", how="left").merge(top_countries, on="topic", how="left")


def write_latex_table(df, path, caption, label, column_format, font_size=r"\scriptsize"):
    table = df.to_latex(
        index=False,
        escape=True,
        longtable=True,
        caption=caption,
        label=label,
        column_format=column_format,
    )
    wrapped = "\n".join([
        r"\begingroup",
        font_size,
        r"\setlength{\tabcolsep}{3pt}",
        r"\renewcommand{\arraystretch}{1.18}",
        table,
        r"\endgroup",
    ]) + "\n"
    path.write_text(wrapped)

# Appendix table 1: substantive higher-order coverage-frame inventory.
frame_col = ANALYSIS_LABEL_COLUMN
frame_summary_cols = [
    frame_col,
    "weighted_articles",
    "weighted_share",
    "raw_topics",
    "countries_ge_5pct",
    "max_country_share",
    "top_countries",
    "top_fine_topics",
    "substantive_description",
    "group_comparability",
]
frame_summary_cols = [col for col in frame_summary_cols if col in substantive_summary.columns]
frame_latex = substantive_summary[frame_summary_cols].copy()
frame_latex = frame_latex.rename(
    columns={
        frame_col: "Coverage frame",
        "weighted_articles": "Weighted articles",
        "weighted_share": "Weighted share",
        "raw_topics": "Raw topics",
        "countries_ge_5pct": "Countries >=5%",
        "max_country_share": "Max country share",
        "top_countries": "Top countries",
        "top_fine_topics": "Original BERTopic topics",
        "substantive_description": "Substantive interpretation",
        "group_comparability": "Comparability",
    }
)
if "Weighted articles" in frame_latex.columns:
    frame_latex["Weighted articles"] = weighted_count_string(frame_latex["Weighted articles"])
if "Weighted share" in frame_latex.columns:
    frame_latex["Weighted share"] = percent_string(frame_latex["Weighted share"], 1)
if "Max country share" in frame_latex.columns:
    frame_latex["Max country share"] = percent_string(frame_latex["Max country share"], 1)

write_latex_table(
    frame_latex,
    LATEX_DIR / "table_topic_coverage_frame_summary.tex",
    caption=(
        "Substantive inventory of higher-order coverage frames in political-corruption news. "
        "Weighted article counts and shares use country-year sampling weights. "
        "Countries >=5% reports the number of countries contributing at least 5 percent "
        "of weighted articles within the frame."
    ),
    label="tab:topic_coverage_frame_summary",
    column_format=r"p{0.12\textwidth}rrrrp{0.08\textwidth}p{0.14\textwidth}p{0.20\textwidth}p{0.28\textwidth}p{0.07\textwidth}",
)

# Appendix table 2: detailed fine-grained subtopic codebook.
topic_weights = weighted_topic_table()
full_topics = topic_lookup[topic_lookup["Topic"].ne(-1)].copy()
full_topics = full_topics.merge(topic_weights, left_on="Topic", right_on="topic", how="left")
full_topic_cols = [
    "Topic",
    "topic_short_label",
    "topic_group_short_label",
    "weighted_articles",
    "weighted_share",
    "top_countries",
    "llm_topic_summary",
    "llm_inclusion_rule",
    "llm_exclusion_rule",
    "llm_country_event_specific",
    "llm_cross_country_comparability",
]
full_topic_cols = [col for col in full_topic_cols if col in full_topics.columns]
full_topics_latex = full_topics[full_topic_cols].sort_values("weighted_share", ascending=False).copy()
if {"weighted_articles", "weighted_share"}.issubset(full_topics_latex.columns):
    full_topics_latex["weighted_size"] = (
        weighted_count_string(full_topics_latex["weighted_articles"])
        + " ("
        + percent_string(full_topics_latex["weighted_share"], 2)
        + ")"
    )
if {"llm_country_event_specific", "llm_cross_country_comparability"}.issubset(full_topics_latex.columns):
    full_topics_latex["scope_and_comparability"] = (
        "Country/event-specific: "
        + full_topics_latex["llm_country_event_specific"].fillna("").astype(str)
        + "; comparability: "
        + full_topics_latex["llm_cross_country_comparability"].fillna("").astype(str)
    )
subtopic_cols = [
    "Topic",
    "topic_short_label",
    "topic_group_short_label",
    "weighted_size",
    "top_countries",
    "llm_topic_summary",
    "llm_inclusion_rule",
    "llm_exclusion_rule",
    "scope_and_comparability",
]
subtopic_cols = [col for col in subtopic_cols if col in full_topics_latex.columns]
full_topics_latex = full_topics_latex[subtopic_cols].rename(
    columns={
        "Topic": "Topic ID",
        "topic_short_label": "Original BERTopic topic",
        "topic_group_short_label": "Coverage frame",
        "weighted_size": "Weighted size",
        "top_countries": "Top countries",
        "llm_topic_summary": "Substantive description",
        "llm_inclusion_rule": "Includes",
        "llm_exclusion_rule": "Excludes",
        "scope_and_comparability": "Scope",
    }
)
write_latex_table(
    full_topics_latex,
    LATEX_DIR / "table_all_topics_llm_coverage_frames.tex",
    caption=(
        "Fine-grained BERTopic subtopic codebook with LLM-generated substantive descriptions. "
        "Weighted size reports weighted articles and weighted corpus share in parentheses. "
        "Top countries reports the largest country contributors within each topic."
    ),
    label="tab:all_topics_llm_coverage_frames",
    column_format=r"rp{0.11\textwidth}p{0.09\textwidth}p{0.07\textwidth}p{0.11\textwidth}p{0.20\textwidth}p{0.15\textwidth}p{0.15\textwidth}p{0.09\textwidth}",
)

print(f"Saved inspection tables under: {EXPORT_DIR}")
print(f"Saved LaTeX tables under: {LATEX_DIR}")
